# Comparing Asset Pricing Models

In this notebook, we will replicate the relevant parts of Table 3 (and eventually Table 4) from Fama and French (2012).

We will estimate three asset pricing models:

1. CAPM
2. Fama-French three-factor model
3. Carhart four-factor model

We will use portfolios formed using company size and book-to-market ratio.

We will study three cases:

1. Global Developed portfolios with Global Developed factors
2. Japanese portfolios with Global Developed factors
3. Japanese portfolios with Japanese factors

For each case, we will first use all 25 portfolios. We will then remove the five portfolios in the smallest size group and repeat the analysis using the remaining 20 portfolios.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

In [2]:
DATA_DIR = Path("cleaned_data")

## Load the factor data

The three-factor files contain:

- `Mkt-RF`: Market return minus the risk-free return
- `SMB`: Return of small companies minus large companies
- `HML`: Return of value companies minus growth companies
- `RF`: Risk-free return

The momentum files contain:

- `WML`: Return of past winners minus past losers

In [3]:
developed_factors = pd.read_csv(
    DATA_DIR / "developed_3_factors.csv",
    parse_dates=["date"]
)

In [4]:
developed_momentum = pd.read_csv(
    DATA_DIR / "developed_momentum.csv",
    parse_dates=["date"]
)

In [5]:
japan_factors = pd.read_csv(
    DATA_DIR / "japan_3_factors.csv",
    parse_dates=["date"]
)

In [6]:
japan_momentum = pd.read_csv(
    DATA_DIR / "japan_momentum.csv",
    parse_dates=["date"]
)

## Load the portfolio data

The portfolio files contain value-weighted monthly returns for 25 portfolios.

The portfolios are formed using:

- Five company-size groups
- Five book-to-market groups

This gives 5 × 5 = 25 portfolios for each market.

In [7]:
developed_portfolios = pd.read_csv(
    DATA_DIR / "developed_25_size_bm.csv",
    parse_dates=["date"]
)

In [8]:
japan_portfolios = pd.read_csv(
    DATA_DIR / "japan_25_size_bm.csv",
    parse_dates=["date"]
)

## Add the momentum factor

The momentum factor is stored in a separate file.

We merge the momentum data with the three-factor data using `date`.

The `one_to_one` check confirms that each month appears only once in each dataset.

In [9]:
developed_factors = developed_factors.merge(
    developed_momentum,
    on="date",
    validate="one_to_one"
)

In [10]:
japan_factors = japan_factors.merge(
    japan_momentum,
    on="date",
    validate="one_to_one"
)

## Check the datasets

The sample runs from November 1990 to March 2011.

Each dataset should contain 245 monthly observations.

The factor datasets should contain six columns:

- `date`
- `Mkt-RF`
- `SMB`
- `HML`
- `RF`
- `WML`

The portfolio datasets should contain one date column and 25 portfolio return columns.

In [11]:
print("Developed factors:", developed_factors.shape)
print("Japanese factors:", japan_factors.shape)
print("Developed portfolios:", developed_portfolios.shape)
print("Japanese portfolios:", japan_portfolios.shape)

Developed factors: (245, 6)
Japanese factors: (245, 6)
Developed portfolios: (245, 26)
Japanese portfolios: (245, 26)


## Select the portfolios

The 5x5 results use all 25 portfolios.

The 4x5 results remove the five portfolios in the smallest size group.

The smallest size group appears in the first five portfolio columns. Removing these columns leaves 20 portfolios.

In [12]:
all_portfolios = developed_portfolios.columns.drop("date").tolist()

In [13]:
without_microcaps = all_portfolios[5:]

In [14]:
print("Number of 5x5 portfolios:", len(all_portfolios))
print("Number of 4x5 portfolios:", len(without_microcaps))

Number of 5x5 portfolios: 25
Number of 4x5 portfolios: 20


## Define the models

The CAPM uses only the market factor.

The three-factor model adds size and value.

The four-factor model also adds momentum.

In [15]:
models = {
    "CAPM": ["Mkt-RF"],
    "Three-factor": ["Mkt-RF", "SMB", "HML"],
    "Four-factor": ["Mkt-RF", "SMB", "HML", "WML"]
}

## Portfolio regressions

A separate regression is estimated for every portfolio.

The dependent variable is the portfolio's excess return:

Portfolio excess return = Portfolio return minus RF

The explanatory variables depend on the model.

### CAPM

The portfolio excess return is explained using `Mkt-RF`.

### Three-factor model

The portfolio excess return is explained using `Mkt-RF`, `SMB` and `HML`.

### Four-factor model

The portfolio excess return is explained using `Mkt-RF`, `SMB`, `HML` and `WML`.

Each regression includes a constant. This constant is the portfolio's alpha.

Alpha is the average monthly return that remains unexplained after accounting for the factors included in the model.

## Market and factor combinations

We will estimate the models for three combinations.

### Global portfolios with Global factors

Global Developed portfolio returns are explained using Global Developed factors.

### Japanese portfolios with Global factors

Japanese portfolio returns are explained using Global Developed factors.

### Japanese portfolios with Japanese factors

Japanese portfolio returns are explained using Japanese factors.

The second and third cases allow us to compare whether Japanese portfolio returns are explained better by Global or Japanese factors.

## Statistics reported in Table 3

Table 3 combines the results from the individual portfolio regressions.

The statistics reported depend on whether the analysis uses 25 or 20 portfolios.

### 5x5 results

The 5x5 results use all 25 portfolios.

The paper reports:

- `GRS`
- `|a|`
- `Adjusted R2`
- `s(a)`
- `SR(a)`

### 4x5 results

The 4x5 results remove the five portfolios in the smallest size group and use the remaining 20 portfolios.

The paper reports only:

- `GRS`
- `|a|`
- `SR(a)`

The paper does not report `Adjusted R2` or `s(a)` for the 4x5 cases.

### GRS statistic

The GRS statistic tests whether all portfolio alphas are jointly equal to zero.

A larger GRS value gives more evidence that the model leaves unexplained returns.

This statistic is not provided directly by the individual regression outputs. We calculate it using:

- The alphas from all portfolio regressions
- The residuals from all portfolio regressions
- The factor returns
- The number of portfolios
- The number of factors
- The number of observations

For the 5x5 case, calculate one GRS statistic using all 25 regressions together.

For the 4x5 case, calculate one GRS statistic using all 20 regressions together.

Do not calculate a separate GRS statistic for each portfolio and then average it.

### Average absolute alpha: |a|

Alpha is the constant reported by each portfolio regression. It measures the average monthly return that the model does not explain.

The alpha can be taken directly from the fitted regression output.

To calculate `|a|`:

1. Collect the alpha from every regression.
2. Take the absolute value of every alpha.
3. Calculate the simple average.

For the 5x5 case, average the 25 absolute alphas.

For the 4x5 case, average the 20 absolute alphas.

A smaller value means that the model leaves less unexplained return.

### Average adjusted R-squared

Adjusted R-squared measures how much of the monthly movement in a portfolio's excess return is explained by the model.

It can be taken directly from each fitted regression output.

For the 5x5 results:

1. Collect the adjusted R-squared from each of the 25 regressions.
2. Calculate the simple average.

A higher average adjusted R-squared means that the model explains more of the variation in portfolio returns.

The paper reports this statistic only for the 5x5 results. It is not reported for the 4x5 results.

### Average standard error of alpha: s(a)

The standard error of alpha shows how precisely the alpha has been estimated.

It can be taken directly from each fitted regression output as the standard error of the constant.

For the 5x5 results:

1. Collect the standard error of alpha from each of the 25 regressions.
2. Calculate the simple average.

A smaller value means that the portfolio alphas are estimated more precisely.

The paper reports this statistic only for the 5x5 results. It is not reported for the 4x5 results.

### Sharpe ratio of the alphas: SR(a)

`SR(a)` measures the combined size of the portfolio alphas relative to the risk in the regression residuals.

To calculate it:

1. Collect the alpha from every portfolio regression.
2. Place the alphas in one vector.
3. Collect the residuals from every regression.
4. Use the residuals to calculate the residual covariance matrix.
5. Combine the alpha vector and residual covariance matrix using the formula provided in the paper.
6. Take the square root of the resulting value.

For the 5x5 case, use all 25 alphas and their residuals.

For the 4x5 case, use the 20 alphas and their residuals.

This produces one joint `SR(a)` value for each model. It is not the average of separate portfolio Sharpe ratios.

The exact formula for `SR(a)` is provided in the paper.

## Producing the Table 3 results

For each task:

1. Select the required portfolio and factor data.
2. Merge the datasets using `date`.
3. Subtract `RF` from every portfolio return.
4. Add a constant to the explanatory variables.
5. Run a separate regression for every selected portfolio.
6. Store the fitted regression results for each model.
7. Use the stored results to calculate the required Table 3 statistics.
8. Place the three models in the rows of one table.
9. Round the final values to two decimal places.

For the 5x5 tasks, report:

- `GRS`
- `|a|`
- `Adjusted R2`
- `s(a)`
- `SR(a)`

For the 4x5 tasks, report only:

- `GRS`
- `|a|`
- `SR(a)`

## Task 1: Global portfolios with Global factors, 5x5

Use:

- `developed_portfolios`
- `developed_factors`
- All 25 portfolios in `all_portfolios`

For each portfolio, run:

1. CAPM
2. Three-factor model
3. Four-factor model

Store the regression results and calculate the five Table 3 statistics.

Print one table containing the results for the three models.

Compare your results with the Global 5x5 part of Table 3. State whether your values match and interpret the main differences between the models.

In [16]:
# Shared helpers, used by every task in this notebook.

T_EXPECTED = 245

FIVE_BY_FIVE = ["GRS", "|a|", "Adjusted R2", "s(a)", "SR(a)"]
FOUR_BY_FIVE = ["GRS", "|a|", "SR(a)"]


def prepare_data(portfolios, factors, portfolio_columns):
    """Merge on date and return the excess portfolio returns and the matched factor data."""
    overlap = portfolios["date"].isin(factors["date"]).sum()
    print("Overlapping dates between portfolios and factors:", overlap)
    if overlap != T_EXPECTED:
        raise ValueError(f"Expected {T_EXPECTED} overlapping dates, found {overlap}. Stop.")

    merged = portfolios.merge(factors, on="date", validate="one_to_one")
    excess = merged[portfolio_columns].sub(merged["RF"], axis=0)
    return excess, merged


def run_model(excess, merged, factor_columns):
    """One OLS regression per portfolio: excess return on a constant plus the model's factors."""
    X = sm.add_constant(merged[factor_columns])
    return {portfolio: sm.OLS(excess[portfolio], X).fit() for portfolio in excess.columns}


def check_results(results, n_portfolios):
    """Mechanical checks: right number of regressions, 245 observations, adjusted R2 in [0, 1]."""
    if len(results) != n_portfolios:
        raise ValueError(f"Expected {n_portfolios} regressions, found {len(results)}")
    for portfolio, result in results.items():
        if result.nobs != T_EXPECTED:
            raise ValueError(f"{portfolio}: {result.nobs} observations, expected {T_EXPECTED}")
        if not 0 <= result.rsquared_adj <= 1:
            raise ValueError(f"{portfolio}: adjusted R2 {result.rsquared_adj:.3f} outside [0, 1]")


def alphas_and_residuals(results):
    """Stack the N alphas into a vector and the residuals into a T x N matrix."""
    alphas = np.array([result.params["const"] for result in results.values()])
    residuals = np.column_stack([result.resid for result in results.values()])
    return alphas, residuals


def residual_covariance(residuals):
    """N x N covariance matrix of the regression residuals (divided by T)."""
    T = residuals.shape[0]
    return residuals.T @ residuals / T


def grs_statistic(results, factor_returns):
    """Gibbons, Ross and Shanken (1989) test that all N alphas are jointly zero.

    GRS = (T - N - K) / N * (a' S^-1 a) / (1 + f' W^-1 f)
    where S is the residual covariance matrix, f the vector of factor means and
    W the factor covariance matrix (both covariances divided by T).
    """
    alphas, residuals = alphas_and_residuals(results)
    T, N = residuals.shape
    K = factor_returns.shape[1]

    S = residual_covariance(residuals)
    f_mean = factor_returns.mean().to_numpy()
    f_demeaned = factor_returns.to_numpy() - f_mean
    W = f_demeaned.T @ f_demeaned / T

    alpha_term = alphas @ np.linalg.solve(S, alphas)
    factor_term = f_mean @ np.linalg.solve(W, f_mean)
    return (T - N - K) / N * alpha_term / (1 + factor_term)


def sr_alpha(results):
    """Sharpe ratio of the intercepts, equation (3) in the paper: SR(a) = (a' S^-1 a)^(1/2)."""
    alphas, residuals = alphas_and_residuals(results)
    S = residual_covariance(residuals)
    return np.sqrt(alphas @ np.linalg.solve(S, alphas))


def model_statistics(results, factor_returns):
    """Table 3 statistics for one model on one set of portfolios."""
    alphas, _ = alphas_and_residuals(results)
    return {
        "GRS": grs_statistic(results, factor_returns),
        "|a|": np.mean(np.abs(alphas)),
        "Adjusted R2": np.mean([result.rsquared_adj for result in results.values()]),
        "s(a)": np.mean([result.bse["const"] for result in results.values()]),
        "SR(a)": sr_alpha(results),
    }


def scorecard(portfolios, factors, portfolio_columns, models, statistics):
    """Run every model on the selected portfolios.

    Returns the fitted regressions (model name -> portfolio -> result) and one
    table with a row per model and the requested statistics as columns.
    """
    excess, merged = prepare_data(portfolios, factors, portfolio_columns)

    fitted = {}
    rows = {}
    for model_name, factor_columns in models.items():
        results = run_model(excess, merged, factor_columns)
        check_results(results, len(portfolio_columns))
        fitted[model_name] = results
        rows[model_name] = model_statistics(results, merged[factor_columns])

    print(f"Checks passed: {len(portfolio_columns)} regressions per model, "
          f"{T_EXPECTED} observations each, adjusted R2 between 0 and 1")
    table = pd.DataFrame(rows).T[statistics]
    return fitted, table.round(2)


# Task 1: Global portfolios, Global factors, 5x5

global_fitted, global_5x5 = scorecard(
    developed_portfolios, developed_factors, all_portfolios, models, FIVE_BY_FIVE
)
print("Global portfolios with Global factors, 5x5")
global_5x5

Overlapping dates between portfolios and factors: 245
Checks passed: 25 regressions per model, 245 observations each, adjusted R2 between 0 and 1
Global portfolios with Global factors, 5x5


,GRS,|a|,Adjusted R2,s(a),SR(a)
CAPM,4.54,0.21,0.81,0.14,0.72
Three-factor,4.11,0.13,0.95,0.07,0.71
Four-factor,3.66,0.11,0.95,0.07,0.69


## Task 2: Global portfolios with Global factors, 4x5

Use:

- `developed_portfolios`
- `developed_factors`
- The 20 portfolios in `without_microcaps`

For each portfolio, run the CAPM, three-factor model and four-factor model.

Store the regression results.

For each model, report the three statistics shown in the 4x5 part of Table 3:

- `GRS`
- `|a|`
- `SR(a)`

Print one table containing the results for the three models.

Compare your results with the Global 4x5 part of Table 3. Discuss whether removing the five smallest portfolios changes the results.

In [17]:
global_4x5_fitted, global_4x5 = scorecard(
    developed_portfolios, developed_factors, without_microcaps, models, FOUR_BY_FIVE
)
print("Global portfolios with Global factors, 4x5 (without microcaps)")
global_4x5

Overlapping dates between portfolios and factors: 245


Checks passed: 20 regressions per model, 245 observations each, adjusted R2 between 0 and 1
Global portfolios with Global factors, 4x5 (without microcaps)


,GRS,|a|,SR(a)
CAPM,2.18,0.17,0.44
Three-factor,2.76,0.10,0.51
Four-factor,2.34,0.08,0.49


## Task 3: Japanese portfolios with Global factors, 5x5

Use:

- `japan_portfolios`
- `developed_factors`
- All 25 portfolios in `all_portfolios`

For each Japanese portfolio, run the CAPM, three-factor model and four-factor model using Global Developed factors.

Store the regression results and calculate the five Table 3 statistics.

Print one table containing the results for the three models.

Compare your results with the Japan, Global factors, 5x5 part of Table 3. Interpret how well Global factors explain Japanese portfolio returns.

In [18]:
japan_global_fitted, japan_global_5x5 = scorecard(
    japan_portfolios, developed_factors, all_portfolios, models, FIVE_BY_FIVE
)
print("Japanese portfolios with Global factors, 5x5")
japan_global_5x5

Overlapping dates between portfolios and factors: 245


Checks passed: 25 regressions per model, 245 observations each, adjusted R2 between 0 and 1
Japanese portfolios with Global factors, 5x5


,GRS,|a|,Adjusted R2,s(a),SR(a)
CAPM,1.48,0.48,0.28,0.39,0.41
Three-factor,1.20,0.71,0.35,0.38,0.38
Four-factor,1.18,0.68,0.35,0.39,0.39


## Task 4: Japanese portfolios with Global factors, 4x5

Use:

- `japan_portfolios`
- `developed_factors`
- The 20 portfolios in `without_microcaps`

For each Japanese portfolio, run the CAPM, three-factor model and four-factor model using Global Developed factors.

Store the regression results.

For each model, report the three statistics shown in the 4x5 part of Table 3:

- `GRS`
- `|a|`
- `SR(a)`

Print one table containing the results for the three models.

Compare your results with the Japan, Global factors, 4x5 part of Table 3. Discuss whether removing the smallest Japanese portfolios changes the results.

In [19]:
japan_global_4x5_fitted, japan_global_4x5 = scorecard(
    japan_portfolios, developed_factors, without_microcaps, models, FOUR_BY_FIVE
)
print("Japanese portfolios with Global factors, 4x5 (without microcaps)")
japan_global_4x5

Overlapping dates between portfolios and factors: 245


Checks passed: 20 regressions per model, 245 observations each, adjusted R2 between 0 and 1
Japanese portfolios with Global factors, 4x5 (without microcaps)


,GRS,|a|,SR(a)
CAPM,1.53,0.52,0.37
Three-factor,1.20,0.73,0.34
Four-factor,1.14,0.71,0.34


## Task 5: Japanese portfolios with Japanese factors, 5x5

Use:

- `japan_portfolios`
- `japan_factors`
- All 25 portfolios in `all_portfolios`

For each Japanese portfolio, run the CAPM, three-factor model and four-factor model using Japanese factors.

Store the regression results and calculate the five Table 3 statistics.

Print one table containing the results for the three models.

Compare your results with the Japan, Local factors, 5x5 part of Table 3. Compare these results with Task 3 and determine whether Japanese factors explain Japanese returns better than Global factors.

In [20]:
japan_local_fitted, japan_local_5x5 = scorecard(
    japan_portfolios, japan_factors, all_portfolios, models, FIVE_BY_FIVE
)
print("Japanese portfolios with Japanese factors, 5x5")
japan_local_5x5

Overlapping dates between portfolios and factors: 245


Checks passed: 25 regressions per model, 245 observations each, adjusted R2 between 0 and 1
Japanese portfolios with Japanese factors, 5x5


,GRS,|a|,Adjusted R2,s(a),SR(a)
CAPM,1.18,0.18,0.78,0.22,0.37
Three-factor,0.93,0.12,0.93,0.12,0.33
Four-factor,0.90,0.11,0.93,0.12,0.33


## Task 6: Japanese portfolios with Japanese factors, 4x5

Use:

- `japan_portfolios`
- `japan_factors`
- The 20 portfolios in `without_microcaps`

For each Japanese portfolio, run the CAPM, three-factor model and four-factor model using Japanese factors.

Store the regression results.

For each model, report the three statistics shown in the 4x5 part of Table 3:

- `GRS`
- `|a|`
- `SR(a)`

Print one table containing the results for the three models.

Compare your results with the Japan, Local factors, 4x5 part of Table 3.

Discuss:

1. Which model performs best.
2. Whether removing the smallest portfolios changes the results.
3. Whether Global or Japanese factors explain Japanese portfolio returns better.

In [21]:
japan_local_4x5_fitted, japan_local_4x5 = scorecard(
    japan_portfolios, japan_factors, without_microcaps, models, FOUR_BY_FIVE
)
print("Japanese portfolios with Japanese factors, 4x5 (without microcaps)")
japan_local_4x5

Overlapping dates between portfolios and factors: 245


Checks passed: 20 regressions per model, 245 observations each, adjusted R2 between 0 and 1
Japanese portfolios with Japanese factors, 4x5 (without microcaps)


,GRS,|a|,SR(a)
CAPM,1.14,0.18,0.32
Three-factor,1.05,0.10,0.31
Four-factor,1.02,0.09,0.31


# Table 4: Individual Portfolio Alphas

Table 3 summarizes the results across all 25 or 20 portfolio regressions.

Table 4 looks inside these summary results. It reports the alpha and alpha t-statistic for each individual portfolio.

Therefore:

- Table 3 tells us whether a model performs well overall.
- Table 4 shows which particular portfolios cause the model to perform well or poorly.

For example, Table 4 can show whether a model has difficulty explaining the returns of small growth, small value, big growth or big value portfolios.

## What Table 4 reports

For each portfolio, Table 4 reports:

- `a`: The regression alpha
- `t(a)`: The t-statistic of the alpha

Both values are taken directly from the fitted regression output.

The alpha is the regression constant.

The t-statistic is calculated as:

Alpha divided by the standard error of alpha.

Unlike Table 3, these values are not averaged across portfolios.

The 25 alphas and 25 t-statistics are arranged in 5 × 5 matrices.

The rows represent company size:

1. Small
2. Size group 2
3. Size group 3
4. Size group 4
5. Big

The columns represent book-to-market:

1. Low
2. Group 2
3. Group 3
4. Group 4
5. High

## Task 7: Global portfolio alphas using Global factors

You have already estimated and stored the regression outputs for the Global portfolios using Global factors in Task 1.

Use the stored 5x5 regression results for all 25 portfolios.

For each of the following models, report:

- The alpha of each portfolio
- The t-statistic of each alpha

Models:

1. CAPM
2. Three-factor model
3. Four-factor model

Arrange the 25 alphas in a 5 × 5 matrix.

Arrange the 25 alpha t-statistics in another 5 × 5 matrix.

Compare your results with the section titled:

`Global size-B/M returns regressed on global factors`

in Table 4.

In [22]:
from IPython.display import display

SIZE_LABELS = ["Small", "2", "3", "4", "Big"]
BM_LABELS = ["Low", "2", "3", "4", "High"]


def alpha_grids(results, column_labels):
    """The 25 alphas and their t-statistics, each arranged 5x5 with rows Small to Big."""
    if len(results) != 25:
        raise ValueError(f"Expected 25 regressions, found {len(results)}")
    alphas = [result.params["const"] for result in results.values()]
    t_stats = [result.tvalues["const"] for result in results.values()]

    alpha_grid = pd.DataFrame(np.reshape(alphas, (5, 5)), index=SIZE_LABELS, columns=column_labels)
    t_grid = pd.DataFrame(np.reshape(t_stats, (5, 5)), index=SIZE_LABELS, columns=column_labels)
    return alpha_grid.round(2), t_grid.round(2)


global_alpha_grids = {}

for model_name, results in global_fitted.items():
    alpha_grid, t_grid = alpha_grids(results, BM_LABELS)
    global_alpha_grids[model_name] = {"a": alpha_grid, "t(a)": t_grid}

    print(f"Global size-B/M returns regressed on global factors: {model_name}")
    display(pd.concat({"a": alpha_grid, "t(a)": t_grid}, axis=1))

Global size-B/M returns regressed on global factors: CAPM


a                          t(a)                        
        Low     2     3     4  High   Low     2     3     4  High
Small -0.43  0.01  0.33  0.38  0.76 -1.84  0.04  1.72  2.53  4.70
2     -0.44 -0.06  0.11  0.27  0.41 -2.17 -0.43  0.88  2.24  2.80
3     -0.33 -0.10  0.08  0.17  0.35 -1.95 -0.70  0.78  1.51  2.61
4     -0.12 -0.01  0.04  0.21  0.25 -0.77 -0.12  0.46  1.93  1.96
Big   -0.15 -0.05  0.05  0.10  0.06 -1.34 -0.80  0.69  1.12  0.42

Global size-B/M returns regressed on global factors: Three-factor


a                          t(a)                        
        Low     2     3     4  High   Low     2     3     4  High
Small -0.29  0.03  0.26  0.17  0.44 -2.95  0.33  3.02  2.48  5.93
2     -0.27 -0.07 -0.03 -0.01  0.01 -3.86 -1.05 -0.65 -0.23  0.25
3     -0.11 -0.11 -0.09 -0.13 -0.05 -1.51 -1.53 -1.25 -1.94 -0.72
4      0.14 -0.08 -0.15 -0.08 -0.14  1.70 -1.12 -2.35 -1.09 -2.15
Big    0.19 -0.01 -0.05 -0.09 -0.26  3.18 -0.09 -0.99 -1.46 -3.04

Global size-B/M returns regressed on global factors: Four-factor


a                          t(a)                        
        Low     2     3     4  High   Low     2     3     4  High
Small -0.31  0.04  0.23  0.18  0.41 -3.02  0.45  2.57  2.56  5.41
2     -0.22 -0.02 -0.02  0.02  0.02 -3.10 -0.24 -0.37  0.31  0.31
3     -0.07 -0.11 -0.06 -0.09 -0.03 -0.95 -1.49 -0.79 -1.27 -0.44
4      0.13 -0.03 -0.12 -0.03 -0.09  1.51 -0.43 -1.85 -0.37 -1.45
Big    0.22 -0.03 -0.05 -0.07 -0.17  3.60 -0.50 -0.81 -1.22 -2.05

## Task 8: Japanese portfolio alphas using Japanese factors

You have already estimated and stored the regression outputs for the Japanese portfolios using Japanese factors in Task 5.

Use the stored 5x5 regression results for all 25 portfolios.

Table 4 reports only the local Japanese three-factor model in this section.

Using the stored three-factor regression results, report:

- The alpha of each portfolio
- The t-statistic of each alpha

Arrange the 25 alphas in a 5 × 5 matrix.

Arrange the 25 alpha t-statistics in another 5 × 5 matrix.

Compare your results with the section titled:

`Japanese size/B-M returns regressed on Japanese factors`

in Table 4.

In [23]:
# Table 4 reports only the local three-factor model for Japan.
alpha_grid, t_grid = alpha_grids(japan_local_fitted["Three-factor"], BM_LABELS)
japan_alpha_grids = {"Three-factor": {"a": alpha_grid, "t(a)": t_grid}}

print("Japanese size-B/M returns regressed on Japanese factors: Three-factor")
display(pd.concat({"a": alpha_grid, "t(a)": t_grid}, axis=1))

Japanese size-B/M returns regressed on Japanese factors: Three-factor


a                          t(a)                        
        Low     2     3     4  High   Low     2     3     4  High
Small  0.16  0.13  0.20  0.19  0.21  0.82  0.91  1.40  1.99  2.14
2     -0.03 -0.17 -0.05  0.08 -0.07 -0.16 -1.61 -0.40  0.90 -1.09
3     -0.13 -0.13 -0.22 -0.16  0.00 -0.80 -1.11 -2.09 -1.69  0.04
4     -0.17 -0.05 -0.12 -0.07 -0.12 -1.17 -0.38 -1.01 -0.59 -1.20
Big    0.13 -0.03 -0.12  0.08  0.09  1.23 -0.28 -1.05  0.69  0.50